# From proteins to a phylogenetic tree

In this session you will:

1. Search a set of proteins for one core gene, using a profile HMM (`hmmsearch`).
2. Pull out the proteins that match.
3. Combine them with reference sequences of the same gene.
4. Align everything (`mafft`).
5. Trim the alignment (`trimal`).
6. Download the trimmed alignment and build a tree on a web server.

**How to use this notebook.** Click a cell and press **Shift + Enter** to run it. Run the cells in order, from top to bottom. Cells that start with `%%bash` run command-line programs. The other code cells are Python.

**Important: Binder sessions are temporary.** If you close the tab, or leave the session idle for about 10 minutes, it shuts down and your files are lost. Download anything you want to keep.

**Input files** (your instructor has put these in the `data/` folder):

| File | What it is |
|---|---|
| `data/query_proteins.faa` | The proteins you will search |
| `data/core_gene.hmm` | A profile HMM for the core gene |
| `data/references.faa` | Reference sequences of the same gene from known organisms |

## Step 0. Check that the software is installed

Each line should print a version number.

In [ ]:
%%bash
hmmsearch -h | sed -n '2p'
mafft --version 2>&1
trimal --version
seqkit version

## Step 1. Look at the input files

A FASTA file stores sequences. Each sequence has a header line that starts with `>`, followed by the sequence itself. `seqkit stats` counts the sequences and summarizes their lengths.

In [ ]:
%%bash
seqkit stats --quiet data/query_proteins.faa data/references.faa
echo
echo "The first protein in the query file:"
seqkit head -n 1 data/query_proteins.faa

A **profile HMM** is a statistical summary of an alignment of many known versions of a gene. For each position, it records which amino acids are common there and how often gaps occur. `hmmsearch` scores every protein against this profile. Proteins that fit the profile well get high scores.

The lines below come from the top of the HMM file. `NAME` is the name of the model, `LENG` is the number of positions in the model, and `NSEQ` is the number of sequences used to build it.

In [ ]:
%%bash
grep -E "^(NAME|DESC|LENG|NSEQ)" data/core_gene.hmm

**Think about it:** How many proteins are you searching? How long is the model? Keep the model length in mind for Step 3.

## Step 2. Search the proteins with the HMM

What each part of the command does:

- `-E 1e-5` reports hits with an E-value of 1e-5 or lower. This is a loose first filter. You will choose a stricter cutoff in Step 3.
- `--tblout` writes a simple table of hits, one line per protein.
- `-o` writes the full, detailed report (useful for looking at individual alignments to the model).
- The last two arguments are the HMM and the proteins to search.

In [ ]:
%%bash
mkdir -p results

hmmsearch \
    -E 1e-5 \
    --tblout results/hmmsearch_hits.tbl \
    -o results/hmmsearch_full_output.txt \
    data/core_gene.hmm \
    data/query_proteins.faa

echo "Number of hits reported:"
awk '!/^#/' results/hmmsearch_hits.tbl | wc -l

## Step 3. Read the results and choose which hits to keep

Two numbers matter most:

- **E-value**: how many hits this good you would expect by chance alone. Smaller is better. 1e-50 is far stronger than 1e-6.
- **Bit score**: how well the protein fits the model. Higher is better. Unlike the E-value, it does not depend on how many proteins you searched.

The table also shows each protein's length. Compare it to the model length (`LENG`) from Step 1. A hit that is much shorter than the model may be a fragment, or a gene that is split into pieces.

In [ ]:
import pandas as pd
from Bio import SeqIO

# Length of every protein in the query file
lengths = {rec.id: len(rec.seq) for rec in SeqIO.parse("data/query_proteins.faa", "fasta")}

# Read the hmmsearch table (lines starting with '#' are comments)
rows = []
with open("results/hmmsearch_hits.tbl") as handle:
    for line in handle:
        if line.startswith("#"):
            continue
        fields = line.split(maxsplit=18)
        rows.append({
            "protein": fields[0],
            "length_aa": lengths.get(fields[0]),
            "evalue": float(fields[4]),
            "bitscore": float(fields[5]),
            "description": fields[18].strip() if len(fields) > 18 else "",
        })

hits = pd.DataFrame(rows, columns=["protein", "length_aa", "evalue", "bitscore", "description"])
hits = hits.sort_values("bitscore", ascending=False).reset_index(drop=True)

if hits.empty:
    print("No hits found. Check the input files, or try a looser E-value in Step 2.")
hits

The plot below shows the bit scores from best to worst. True matches often sit well above the rest, with a clear drop between them and the weak hits. That drop is a useful place to set your cutoff.

In [ ]:
import matplotlib.pyplot as plt

if not hits.empty:
    ax = hits["bitscore"].plot(marker="o", figsize=(7, 3))
    ax.set_xlabel("Hit rank (best first)")
    ax.set_ylabel("Bit score")
    plt.show()

Now choose your cutoff. Change `EVALUE_CUTOFF` if you want, then run the cell again. The IDs of the hits you keep are saved to `results/hit_ids.txt`.

In [ ]:
EVALUE_CUTOFF = 1e-10   # change this and re-run the cell

keep = hits[hits["evalue"] <= EVALUE_CUTOFF]
keep["protein"].to_csv("results/hit_ids.txt", index=False, header=False)

print(f"Keeping {len(keep)} of {len(hits)} hits (E-value <= {EVALUE_CUTOFF})")
keep

## Step 4. Pull out the matching proteins

This cell does four things, one per line:

1. `seqkit grep` copies the sequences listed in `hit_ids.txt` out of the query file.
2. `seqkit seq -i` keeps only the first word of each header (the ID) and drops the rest.
3. `seqkit replace` swaps any unusual character for `_`. Tree files break on spaces and on characters such as `| : ( ) , ;`, so clean names avoid problems later.
4. The last `seqkit replace` adds `QUERY_` to the start of each name, so your proteins are easy to spot in the final tree.

In [ ]:
%%bash
seqkit grep -f results/hit_ids.txt data/query_proteins.faa \
    | seqkit seq -i \
    | seqkit replace -p '[^A-Za-z0-9_.-]' -r '_' \
    | seqkit replace -p '^' -r 'QUERY_' \
    > results/hits.faa

seqkit stats results/hits.faa
echo
echo "Names of the extracted proteins:"
grep ">" results/hits.faa

## Step 5. Combine your hits with the reference sequences

The reference names get the same cleaning. Then both files are joined into one. The last command lists any name that appears twice. It should print nothing, because every sequence in an alignment needs a unique name.

In [ ]:
%%bash
seqkit seq -i data/references.faa \
    | seqkit replace -p '[^A-Za-z0-9_.-]' -r '_' \
    > results/references_clean.faa

cat results/references_clean.faa results/hits.faa > results/combined.faa
seqkit stats results/combined.faa

echo
echo "Duplicate names (this should be empty):"
seqkit seq -n results/combined.faa | sort | uniq -d

## Step 6. Align the sequences with MAFFT

An alignment lines up the sequences so that residues in the same column share a common ancestor. Gaps (`-`) are inserted where a sequence has lost or gained residues.

`--auto` lets MAFFT pick a method based on the size of the data. For small datasets it uses its slow but accurate method (L-INS-i). The cell prints which method it chose. MAFFT's progress messages go to `results/mafft.log`.

In [ ]:
%%bash
mafft --auto results/combined.faa > results/combined.aln.faa 2> results/mafft.log

grep -A 1 "Strategy" results/mafft.log
seqkit stats results/combined.aln.faa

*Note for later:* if you already have an alignment of the references, you can add new sequences to it without re-aligning the references:

`mafft --add results/hits.faa --keeplength references_aligned.faa > results/combined.aln.faa`

`--keeplength` keeps the reference alignment columns fixed. Any extra residues in the new sequences that do not fit those columns are removed.

## Step 7. Look at the alignment

This prints a window of the alignment. Change `START` to move along it. Look for columns where most sequences agree, and for regions that are mostly gaps.

In [ ]:
from Bio import AlignIO

START = 0     # first column to show (try 200, 400, ...)
WIDTH = 80    # number of columns to show

aln = AlignIO.read("results/combined.aln.faa", "fasta")
print(f"{len(aln)} sequences, {aln.get_alignment_length()} columns\n")
for rec in aln:
    print(f"{rec.id[:30]:30s} {str(rec.seq[START:START + WIDTH])}")

## Step 8. Trim the alignment with trimAl

Columns that are mostly gaps, or poorly aligned, add noise to the tree. Trimming removes them.

`-gt 0.1` keeps a column only if at least 10% of the sequences have a residue there. In other words, it removes columns where more than 90% of the sequences have a gap. Other common choices are `-gappyout` and `-automated1`. Try them and compare how many columns remain.

`-htmlout` writes a report that shows which columns were kept and which were removed.

In [ ]:
%%bash
trimal -in results/combined.aln.faa \
       -out results/combined.trimmed.faa \
       -gt 0.1 \
       -htmlout results/trimming_report.html

echo "Before trimming (max_len = number of columns):"
seqkit stats results/combined.aln.faa
echo
echo "After trimming:"
seqkit stats results/combined.trimmed.faa

To see the trimming report, open the `results` folder in the file browser on the left and double-click `trimming_report.html`. If it does not display properly, click **Trust HTML** at the top.

The table below shows how much of each sequence is left after trimming. Sequences near the top of the table are mostly gaps.

In [ ]:
from Bio import SeqIO
import pandas as pd

rows = []
for rec in SeqIO.parse("results/combined.trimmed.faa", "fasta"):
    seq = str(rec.seq)
    residues = len(seq) - seq.count("-")
    rows.append({
        "sequence": rec.id,
        "residues_kept": residues,
        "percent_filled": round(100 * residues / len(seq), 1),
    })

pd.DataFrame(rows).sort_values("percent_filled").reset_index(drop=True)

**Think about it:** Is any sequence mostly gaps? Why might that be (a fragment, a split gene, a wrong hit)? Should it stay in the tree?

## Step 9. Download the trimmed alignment

In the file browser on the left, open the `results` folder, right-click **`combined.trimmed.faa`**, and choose **Download**.

This is the file you will upload to the tree-building server.

## Step 10. Build the tree on a web server

Your instructor will tell you which server to use. On any of them:

1. Upload `combined.trimmed.faa` (it is already aligned, so skip any alignment step if the server offers one).
2. Make sure the sequence type is **protein / amino acid**.
3. Let the server choose the substitution model automatically.
4. Turn on branch support (for example, bootstrap).
5. When the job finishes, download the tree in **Newick** format (often a `.treefile` or `.nwk` file).

## Backup plan: if the web server is down

FastTree is installed in this session and builds a tree from a small alignment in seconds. It is less thorough than the web-server methods, but it is fine for a first look. `-lg` uses the LG model of protein evolution. The numbers on the branches are local support values (0 to 1).

In [ ]:
%%bash
FastTree -lg results/combined.trimmed.faa > results/fasttree.nwk 2> results/fasttree.log
echo "Tree written to results/fasttree.nwk"

## Optional: draw your tree here

To view the tree from the web server, drag the downloaded Newick file into the `results` folder in the file browser, then set `TREE_FILE` to its name. To view the FastTree backup tree, leave `TREE_FILE` as it is.

Your sequences (`QUERY_`) are shown in red. The tree is rooted at its midpoint for display only.

In [ ]:
from Bio import Phylo
import matplotlib.pyplot as plt

TREE_FILE = "results/fasttree.nwk"   # change to your downloaded tree, e.g. "results/my_tree.treefile"

tree = Phylo.read(TREE_FILE, "newick")
tree.root_at_midpoint()
tree.ladderize()

height = max(3, 0.35 * tree.count_terminals())
fig, ax = plt.subplots(figsize=(9, height))
Phylo.draw(
    tree,
    axes=ax,
    do_show=False,
    label_colors=lambda name: "red" if name.startswith("QUERY_") else "black",
)
plt.show()

## Questions to discuss

- Where do your `QUERY_` sequences fall in the tree? Which references are their closest relatives?
- How strong is the branch support for that placement?
- Does the placement change if you use a different E-value cutoff (Step 3) or a different trimming setting (Step 8)?